## **Diffusion Models (DDPM)**

**Intuition**

A diffusion model learns to generate data by **reversing a gradual noising process**. We take a clean image $x_0$ and progressively add Gaussian noise over $T$ steps until it becomes pure noise $x_T \approx \mathcal{N}(0, I)$. A neural network is then trained to **undo** one step of this corruption. At sampling time we start from random noise and run the learned reverse process step-by-step to produce a clean sample.

The key idea: denoising a tiny bit at a time is a much easier task than generating a full image in one shot.

### **Forward (noising) process**

The forward process is a fixed (no learnable parameters) Markov chain that adds noise according to a variance schedule $\beta_1, \dots, \beta_T$:

$$
q(x_t \mid x_{t-1}) = \mathcal{N}\big(x_t;\ \sqrt{1-\beta_t}\, x_{t-1},\ \beta_t I\big)
$$

A crucial property is that we can jump to **any** timestep $t$ in closed form. With $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$:

$$
q(x_t \mid x_0) = \mathcal{N}\big(x_t;\ \sqrt{\bar{\alpha}_t}\, x_0,\ (1-\bar{\alpha}_t) I\big)
\quad\Longrightarrow\quad
x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\,\epsilon,\quad \epsilon \sim \mathcal{N}(0, I)
$$

This lets us train on a random $t$ without simulating the whole chain.

### **Reverse (denoising) process and training objective**

The reverse process is what we **learn**:

$$
p_\theta(x_{t-1} \mid x_t) = \mathcal{N}\big(x_{t-1};\ \mu_\theta(x_t, t),\ \Sigma_\theta(x_t, t)\big)
$$

In DDPM, instead of predicting the mean directly, the network $\epsilon_\theta(x_t, t)$ is trained to **predict the noise** $\epsilon$ that was added. After expanding the variational bound and dropping weighting terms, the objective simplifies to a plain MSE between the true and predicted noise:

$$
\mathcal{L}_{\text{simple}} = \mathbb{E}_{x_0,\ \epsilon,\ t}\Big[\ \big\| \epsilon - \epsilon_\theta(\sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\,\epsilon,\ t) \big\|^2\ \Big]
$$

The network is typically a **time-conditioned U-Net**.

### **DDPM training step (pseudocode)**

```python
import torch
import torch.nn.functional as F

# alphas_bar: precomputed cumulative product of (1 - beta_t), shape [T]
def training_step(model, x0, alphas_bar, T):
    B = x0.shape[0]

    # 1. sample a random timestep for each image in the batch
    t = torch.randint(0, T, (B,), device=x0.device)

    # 2. sample noise and form the noised image x_t in closed form
    eps = torch.randn_like(x0)
    a_bar = alphas_bar[t].view(-1, 1, 1, 1)        # broadcast over C,H,W
    x_t = a_bar.sqrt() * x0 + (1 - a_bar).sqrt() * eps

    # 3. predict the noise and minimize MSE to the true noise
    eps_pred = model(x_t, t)
    loss = F.mse_loss(eps_pred, eps)
    return loss
```

### **Sampling (reverse process)**

Start from $x_T \sim \mathcal{N}(0, I)$ and iterate from $t = T$ down to $1$:

$$
x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}}\, \epsilon_\theta(x_t, t)\right) + \sigma_t z,\qquad z \sim \mathcal{N}(0, I)
$$

where $z = 0$ at the final step. Each step removes a little of the predicted noise and re-injects a small amount of fresh noise (except at $t=1$). Faster deterministic samplers such as **DDIM** skip steps to generate in far fewer iterations.

### **Connection to score matching**

Diffusion models are tightly linked to **denoising score matching**. The *score* of the noised distribution is its gradient of log-density, $\nabla_{x_t} \log q(x_t)$. For the Gaussian forward process,

$$
\nabla_{x_t} \log q(x_t \mid x_0) = -\frac{x_t - \sqrt{\bar{\alpha}_t}\, x_0}{1-\bar{\alpha}_t} = -\frac{\epsilon}{\sqrt{1-\bar{\alpha}_t}}
$$

So predicting the noise $\epsilon_\theta$ is **equivalent (up to scaling) to estimating the score**. This is the view used by score-based generative models (Song & Ermon), where sampling is done by Langevin dynamics / solving a reverse-time SDE. DDPM and score matching are two formulations of the same underlying model.

### **Latent diffusion (brief note)**

Running diffusion directly on high-resolution pixels is expensive. **Latent Diffusion Models** (the basis of Stable Diffusion) first compress the image into a lower-dimensional latent space with a pretrained autoencoder (VAE), run the entire diffusion process **in that latent space**, and decode back to pixels at the end. This dramatically reduces compute. Conditioning (e.g. text prompts) is injected into the U-Net via **cross-attention**.

### **References**

- Ho et al., *Denoising Diffusion Probabilistic Models* (DDPM), 2020
- Song & Ermon, *Generative Modeling by Estimating Gradients of the Data Distribution* (score matching), 2019
- Song et al., *Denoising Diffusion Implicit Models* (DDIM), 2021
- Rombach et al., *High-Resolution Image Synthesis with Latent Diffusion Models*, 2022

[The Breakthrough Behind Modern AI Image Generators](https://www.youtube.com/watch?v=1pgiu--4W3I)